In [2]:
from google.colab import files
import os

uploaded = files.upload()

SA_FILE = list(uploaded.keys())[0]

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SA_FILE
os.environ["GOOGLE_CLOUD_PROJECT"] = "tts-eval-489108"
print("Using credentials:", SA_FILE)

Saving tts-eval-489108-14cd0038a031.json to tts-eval-489108-14cd0038a031.json
Using credentials: tts-eval-489108-14cd0038a031.json


In [3]:
TESTS = [
    ("hi-IN", "hi", "native", "general", "आज मौसम साफ है और हम खेत में काम कर रहे हैं।"),
    ("hi-IN", "hi", "native", "agri", "गेहूं की फसल में पीले धब्बे दिख रहे हैं, क्या दवा डालनी चाहिए?"),
    ("hi-IN", "hi", "native", "numbers", "आज की तारीख 27/02/2026 है और मेरा मोबाइल नंबर 9876543210 है।"),
    ("hi-IN", "hi", "roman",  "roman_numbers", "Aaj ki tareekh 27/02/2026 hai aur mera mobile number 9876543210 hai."),

    ("pa-IN", "pa", "native", "general", "ਅੱਜ ਮੌਸਮ ਚੰਗਾ ਹੈ ਅਤੇ ਅਸੀਂ ਖੇਤ ਵਿੱਚ ਕੰਮ ਕਰ ਰਹੇ ਹਾਂ।"),
    ("pa-IN", "pa", "native", "numbers", "ਅੱਜ ਦੀ ਤਾਰੀਖ 27/02/2026 ਹੈ ਤੇ ਮੇਰਾ ਮੋਬਾਈਲ ਨੰਬਰ 9876543210 ਹੈ।"),
    ("pa-IN", "pa", "roman",  "roman_numbers", "Ajj di tareekh 27/02/2026 ae te mera mobile number 9876543210 ae."),

    ("bn-IN", "bn", "native", "general", "আজ আবহাওয়া ভালো এবং আমরা মাঠে কাজ করছি।"),
    ("bn-IN", "bn", "native", "numbers", "আজকের তারিখ 27/02/2026 এবং আমার মোবাইল নম্বর 9876543210।"),
    ("bn-IN", "bn", "roman",  "roman_numbers", "Aajker tarikh 27/02/2026 ebong amar mobile number 9876543210."),

    ("gu-IN", "gu", "native", "general", "આજે હવામાન સારું છે અને અમે ખેતરમાં કામ કરી રહ્યા છીએ।"),
    ("gu-IN", "gu", "native", "numbers", "આજની તારીખ 27/02/2026 છે અને મારો મોબાઇલ નંબર 9876543210 છે।"),
    ("gu-IN", "gu", "roman",  "roman_numbers", "Aajni tarik 27/02/2026 chhe ane maro mobile number 9876543210 chhe."),

    ("ta-IN", "ta", "native", "general", "இன்று வானிலை நல்லதாக உள்ளது, நாங்கள் வயலில் வேலை செய்கிறோம்."),
    ("ta-IN", "ta", "native", "numbers", "இன்றைய தேதி 27/02/2026 மற்றும் என் கைபேசி எண் 9876543210."),
    ("ta-IN", "ta", "roman",  "roman_numbers", "Inraiya thethi 27/02/2026; en mobile number 9876543210."),

    ("hi-IN", "mix", "mixed", "hinglish", "Kal mandi rate check karna hai, please update kar dena."),
]

In [5]:
from google.cloud import texttospeech
from google.cloud.texttospeech import SsmlVoiceGender
from collections import defaultdict

client = texttospeech.TextToSpeechClient()

TEST_LANGS = sorted(set([t[0] for t in TESTS]))
voices = client.list_voices().voices

def is_wavenet(name: str):
    return "-Wavenet-" in name or "-WaveNet-" in name

wavenet_by_lang = defaultdict(list)

for v in voices:
    if not is_wavenet(v.name):
        continue
    for lc in v.language_codes:
        if lc in TEST_LANGS:
            wavenet_by_lang[lc].append(v)

print("WaveNet voice availability:\n")
for lc in TEST_LANGS:
    vlist = wavenet_by_lang.get(lc, [])
    print(lc, "->", len(vlist), "voices")
    for v in vlist[:5]:
        gender = str(v.ssml_gender).split(".")[-1]
        print("   ", v.name, f"({gender})")

WaveNet voice availability:

bn-IN -> 4 voices
    bn-IN-Wavenet-A (2)
    bn-IN-Wavenet-B (1)
    bn-IN-Wavenet-C (2)
    bn-IN-Wavenet-D (1)
gu-IN -> 4 voices
    gu-IN-Wavenet-A (2)
    gu-IN-Wavenet-B (1)
    gu-IN-Wavenet-C (2)
    gu-IN-Wavenet-D (1)
hi-IN -> 6 voices
    hi-IN-Wavenet-A (2)
    hi-IN-Wavenet-B (1)
    hi-IN-Wavenet-C (1)
    hi-IN-Wavenet-D (2)
    hi-IN-Wavenet-E (2)
pa-IN -> 4 voices
    pa-IN-Wavenet-A (2)
    pa-IN-Wavenet-B (1)
    pa-IN-Wavenet-C (2)
    pa-IN-Wavenet-D (1)
ta-IN -> 4 voices
    ta-IN-Wavenet-A (2)
    ta-IN-Wavenet-B (1)
    ta-IN-Wavenet-C (2)
    ta-IN-Wavenet-D (1)


In [6]:
picked_wavenet = {}

for lc in TEST_LANGS:
    vlist = wavenet_by_lang.get(lc, [])

    if not vlist:
        picked_wavenet[lc] = None
        continue

    pref = [SsmlVoiceGender.NEUTRAL, SsmlVoiceGender.FEMALE, SsmlVoiceGender.MALE]
    chosen = None

    for g in pref:
        for v in vlist:
            if v.ssml_gender == g:
                chosen = v
                break
        if chosen:
            break

    if not chosen:
        chosen = vlist[0]

    picked_wavenet[lc] = chosen.name

print("Selected WaveNet voices:\n")
for lc in TEST_LANGS:
    print(lc, "->", picked_wavenet[lc])

Selected WaveNet voices:

bn-IN -> bn-IN-Wavenet-A
gu-IN -> gu-IN-Wavenet-A
hi-IN -> hi-IN-Wavenet-A
pa-IN -> pa-IN-Wavenet-A
ta-IN -> ta-IN-Wavenet-A


In [9]:
import time
import pandas as pd
from pathlib import Path

OUT_DIR = Path("/content/cloud_wavenet_baseline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def synthesize_mp3(lang_code, text, voice_name):
    voice = texttospeech.VoiceSelectionParams(
        language_code=lang_code,
        name=voice_name
    )
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )
    inp = texttospeech.SynthesisInput(text=text)

    t0 = time.perf_counter()
    resp = client.synthesize_speech(
        input=inp,
        voice=voice,
        audio_config=audio_config
    )
    latency = time.perf_counter() - t0
    return resp.audio_content, latency

rows = []

print("Running WaveNet baseline...\n")

for (lang_code, lang_tag, script_type, case_type, text) in TESTS:

    voice_name = picked_wavenet.get(lang_code)

    if not voice_name:
        rows.append({
            "lang": lang_code,
            "script": script_type,
            "case": case_type,
            "voice": None,
            "latency_sec": None,
            "file": None,
            "error": "No WaveNet voice available"
        })
        continue

    try:
        audio, latency = synthesize_mp3(lang_code, text, voice_name)

        fname = f"wavenet__{lang_code}__{script_type}__{case_type}.mp3".replace("/", "-")
        fpath = OUT_DIR / fname

        with open(fpath, "wb") as f:
            f.write(audio)

        rows.append({
            "lang": lang_code,
            "script": script_type,
            "case": case_type,
            "voice": voice_name,
            "latency_sec": round(latency, 3),
            "file": str(fpath),
            "error": ""
        })

        print(f"✔ {lang_code} | {case_type} | {round(latency,3)}s")

    except Exception as e:
        rows.append({
            "lang": lang_code,
            "script": script_type,
            "case": case_type,
            "voice": voice_name,
            "latency_sec": None,
            "file": None,
            "error": str(e)[:200]
        })

        print(f"✘ {lang_code} | {case_type} | ERROR")


Running WaveNet baseline...

✔ hi-IN | general | 0.132s
✔ hi-IN | agri | 0.131s
✔ hi-IN | numbers | 0.246s
✔ hi-IN | roman_numbers | 0.389s
✔ pa-IN | general | 0.133s
✔ pa-IN | numbers | 0.246s
✔ pa-IN | roman_numbers | 0.307s
✔ bn-IN | general | 0.115s
✔ bn-IN | numbers | 0.27s
✔ bn-IN | roman_numbers | 0.287s
✔ gu-IN | general | 0.227s
✔ gu-IN | numbers | 0.245s
✔ gu-IN | roman_numbers | 0.246s
✔ ta-IN | general | 0.138s
✔ ta-IN | numbers | 0.269s
✔ ta-IN | roman_numbers | 0.375s
✔ hi-IN | hinglish | 0.173s
